# 01 — Análisis exploratorio del GTFS de la EMT Madrid

**TFM BusFreq Madrid — Fase 1 (Datos y análisis exploratorio).**

Este notebook explora la **topología de la red** de la EMT a partir del GTFS
(`data/raw/gtfs_emt.zip`), descargado del portal de datos abiertos del CRTM.
No requiere claves de API: es el primer entregable tangible de la Fase 1.

El GTFS contiene:

| Archivo | Contenido |
|---|---|
| `routes.txt` | Líneas de bus (route_short_name = número de línea) |
| `stops.txt` | Paradas con coordenadas (lat/lon) |
| `trips.txt` | Viajes; enlaza línea ↔ servicio ↔ recorrido |
| `stop_times.txt` | Paso por cada parada (el más grande, ~100 MB) |
| `frequencies.txt` | **Cadencias nominales (headway) por tramo horario** ← base del optimizador |
| `calendar.txt` | Tipos de servicio: LA=laborable, SA=sábado, FE=festivo |
| `shapes.txt` | Geometría de los recorridos (para el mapa Leaflet) |


In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Ruta al GTFS (relativa a la raíz del proyecto)
GTFS_ZIP = Path("..") / "data" / "raw" / "gtfs_emt.zip"
assert GTFS_ZIP.exists(), f"No encuentro {GTFS_ZIP.resolve()}"
print("GTFS:", GTFS_ZIP.resolve())

In [ ]:
def load_gtfs(name: str, **kwargs) -> pd.DataFrame:
    """Carga un archivo .txt del GTFS directamente desde el zip."""
    with zipfile.ZipFile(GTFS_ZIP) as z:
        with z.open(name) as f:
            return pd.read_csv(f, dtype=str, **kwargs)

routes = load_gtfs("routes.txt")
stops = load_gtfs("stops.txt")
trips = load_gtfs("trips.txt")
calendar = load_gtfs("calendar.txt")
frequencies = load_gtfs("frequencies.txt")

# Tipos numéricos donde toca
stops["stop_lat"] = stops["stop_lat"].astype(float)
stops["stop_lon"] = stops["stop_lon"].astype(float)
frequencies["headway_secs"] = frequencies["headway_secs"].astype(int)

print(f"routes:      {len(routes):>7,}")
print(f"stops:       {len(stops):>7,}")
print(f"trips:       {len(trips):>7,}")
print(f"frequencies: {len(frequencies):>7,}")
print(f"calendar:    {len(calendar):>7,}")

## 1. Líneas de la red

Cuántas líneas tiene la EMT y cómo se identifican.

In [ ]:
print(f"Total de líneas: {routes['route_short_name'].nunique()}")
routes[["route_id", "route_short_name", "route_long_name"]].head(15)

## 2. Tipos de servicio (calendario)

La EMT define el servicio por tipo de día. Esto es clave para tu TFM: las
frecuencias actuales ya distinguen laborable / sábado / festivo, pero **no**
distinguen clima ni eventos — ahí está tu oportunidad de mejora.

In [ ]:
calendar

## 3. Frecuencias nominales (headways) — base del optimizador

`frequencies.txt` da, por viaje y tramo horario, el intervalo entre buses
(`headway_secs`). Esta es la **"frecuencia actual"** contra la que el Módulo 2
comparará sus recomendaciones. La enriquecemos con la línea de cada viaje.

In [ ]:
# Enlazar cada viaje con su línea y tipo de servicio
trip_line = trips[["trip_id", "route_id", "service_id"]].merge(
    routes[["route_id", "route_short_name"]], on="route_id", how="left"
)
freq = frequencies.merge(trip_line, on="trip_id", how="left")
freq["headway_min"] = freq["headway_secs"] / 60

# Hora de inicio del tramo (los GTFS pueden pasar de 24h, ej. 25:00)
freq["start_hour"] = freq["start_time"].str.split(":").str[0].astype(int)

print("Estadísticos de cadencia (minutos entre buses):")
display(freq["headway_min"].describe())
freq[["route_short_name", "service_id", "start_time", "end_time", "headway_min"]].head(10)

In [ ]:
# Cadencia media por hora del día (servicio laborable 'LA')
la = freq[freq["service_id"] == "LA"]
by_hour = la.groupby("start_hour")["headway_min"].mean()

ax = by_hour.plot(kind="bar", figsize=(11, 4), color="#0178BC")
ax.set_title("Cadencia media por hora — servicio laborable (menor = más frecuente)")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Headway medio (min)")
plt.tight_layout()
plt.show()

## 4. Variabilidad de frecuencia por línea → candidatas al piloto

El piloto del TFM son **10–15 líneas** con alta variabilidad de demanda. Como
proxy inicial (antes de tener validaciones reales), miramos qué líneas más
varían su cadencia a lo largo del día: una señal de demanda muy desigual.

In [ ]:
var_by_line = (
    la.groupby("route_short_name")["headway_min"]
    .agg(["min", "max", "mean", "std"])
    .dropna()
    .sort_values("std", ascending=False)
)
var_by_line["rango"] = var_by_line["max"] - var_by_line["min"]
print("Top 15 líneas por variabilidad de cadencia (candidatas al piloto):")
var_by_line.head(15)

## 5. Distribución geográfica de paradas

Mapa rápido (scatter lon/lat) de las ~4.700 paradas. En la Fase 4 esto se
convertirá en un mapa interactivo de calor con Leaflet en el dashboard.

In [ ]:
# Filtrar solo paradas (location_type vacío o '0'), no estaciones padre
paradas = stops[stops["location_type"].isna() | (stops["location_type"] == "0")]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(paradas["stop_lon"], paradas["stop_lat"], s=2, alpha=0.4, color="#0178BC")
ax.set_title(f"Paradas de la EMT Madrid (n={len(paradas):,})")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 6. Conclusiones y siguientes pasos

**Lo que ya sabemos de la red (sin tocar la API):**
- Nº de líneas, paradas y viajes de la EMT.
- Las frecuencias actuales solo distinguen tipo de día (LA/SA/FE), no clima ni eventos.
- Una primera lista de líneas candidatas al piloto por variabilidad de cadencia.

**Pendiente (necesita la API de EMT):**
- Cruzar esta red con las **validaciones históricas** reales por parada/franja.
- Refinar la selección del piloto con variabilidad de *demanda* (no solo de cadencia).

**Para la memoria:** las cifras y gráficas de este notebook alimentan directamente
el capítulo *"Datos y análisis exploratorio"*. Guarda las figuras que te interesen
con `plt.savefig(...)`.